Data Processing for Tethered Balloon

In [ ]:
# imports
import math
import numpy as np
import pandas as pd
from scipy.fft import fft
import plotly.graph_objects as go

In [179]:
# Constants + Strain Gauge Calibration
offset_0 = 406.62
offset_1 = -466.48
offset_2 = 246.45
offset_3 = 313.22

scale_0 = 0.001
scale_1 = 0.001
scale_2 = 0.001
scale_3 = 0.001

duty_cycle = 5

net_or_not = "no_net"

data_folder = f"../four_tether_{net_or_not}_7_1"
filename = f"{data_folder}/four_tether_{duty_cycle}%_{net_or_not}.csv"
print(f"Loading data from: {filename}")

Loading data from: ../four_tether_no_net_7_1/four_tether_5%_no_net.csv


In [180]:
def load_tether_data(filepath):
    """Load CSV data and apply calibration and alignment."""
    
    df = pd.read_csv(filepath, header=None, names=['time', 'raw0', 'raw1', 'raw2', 'raw3'])
    
    df['T0'] = ((df['raw0'] * scale_0) + offset_0) * math.sin(math.radians(55))
    df['T1'] = ((df['raw1'] * scale_1) + offset_1) * math.sin(math.radians(55))
    df['T2'] = ((df['raw2'] * scale_2) + offset_2) * math.sin(math.radians(55))
    df['T3'] = ((df['raw3'] * scale_3) + offset_3) * math.sin(math.radians(55))
    
    for col in ['T0', 'T1', 'T2', 'T3']:
        df[col] = df[col] - df[col].min()

    df['time'] = (df['time'] - df['time'][0])/1000

    df = df[['time', 'T0', 'T1', 'T2', 'T3']]
    
    return df

In [181]:
# Load and calibrate data
df = load_tether_data(filename)
print(f"Loaded data for {duty_cycle}% duty cycle, {net_or_not}")
print(df.head())

Loaded data for 5% duty cycle, no_net
    time         T0        T1         T2        T3
0  0.000  13.099879  0.070447   9.788867  1.291803
1  0.108  12.876251  0.074543  10.012495  1.282792
2  0.216  12.703410  0.091745  10.263975  1.380271
3  0.324  12.456845  0.061436  10.430263  1.412218
4  0.432  12.126727  0.158915  10.576891  1.417952


In [ ]:
# save calibrated data to csv!

output_folder = "../data_indoors"
output_file_name = f"calibrated_four_tether_{duty_cycle}%_{net_or_not}.csv"

output_path = f"{output_folder}/{output_file_name}"
df.to_csv(output_path, index=False)

In [182]:
def plot_tether_tension(df, title=None):
    """Plot tether tension data over time."""
    
    import plotly.graph_objects as go
    
    fig = go.Figure()
    
    tension_columns = [col for col in df.columns if col != 'time']
    
    for col in tension_columns:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[col],
            mode='lines',
            name=col,
            line=dict(width=2)
        ))
    
    fig.update_layout(
        title=title or f"{duty_cycle}% Duty Cycle - {net_or_not.replace('_', ' ').title()}",
        xaxis_title='Time (s)',
        yaxis_title='Tension (g)',
        template='plotly_white',
        hovermode='x unified',
        height=500,
        width=1000
    )
    
    fig.show()

In [183]:
# Plot the tether tension data
plot_tether_tension(df)

Find Vortex Shedding Frequency and Analyze

In [184]:
tether_cols = ['T0', 'T1', 'T2', 'T3']

def fft_for_segment(seg_df, col):
    time = seg_df['time'].values
    data = seg_df[col].values

    data = data - np.mean(data)

    N = len(data)
    if N < 2:
        return np.array([]), np.array([])

    dt = np.mean(np.diff(time))
    if dt <= 0:
        return np.array([]), np.array([])

    fft_vals = fft(data)
    freq = np.fft.fftfreq(N, dt)[:N // 2]
    mag = np.abs(fft_vals[:N // 2]) * 2 / N

    return freq, mag


num_bins = 10

min_time = df['time'].iloc[0]
max_time = df['time'].iloc[-1]
bin_edges = np.linspace(min_time, max_time, num_bins + 1)

spectra_by_tether = {col: [] for col in tether_cols}
freq_axis = None

for i in range(num_bins):
    start_time = bin_edges[i]
    end_time = bin_edges[i + 1]

    if i == num_bins - 1:
        seg_df = df[(df['time'] >= start_time) & (df['time'] <= end_time)].copy()
    else:
        seg_df = df[(df['time'] >= start_time) & (df['time'] < end_time)].copy()

    if len(seg_df) < 10:
        continue

    for col in tether_cols:
        freq, mag = fft_for_segment(seg_df, col)

        if len(freq) == 0:
            continue

        if freq_axis is None:
            freq_axis = freq

        mag_interp = np.interp(freq_axis, freq, mag)
        spectra_by_tether[col].append(mag_interp)


avg_spectra = {}

for col in tether_cols:
    if spectra_by_tether[col]:
        avg_spectra[col] = np.mean(np.vstack(spectra_by_tether[col]), axis=0)


fig = go.Figure()

for col in tether_cols:
    if col in avg_spectra:
        fig.add_trace(go.Scatter(
            x=freq_axis,
            y=avg_spectra[col],
            mode='lines',
            name=col,
            line=dict(width=2)
        ))

fig.update_layout(
    title=f'FFT Bin Averaged Plot - {duty_cycle}% Duty Cycle, {net_or_not.replace("_", " ").title()}',
    xaxis_title='Frequency (Hz)',
    yaxis_title='Average Magnitude',
    template='plotly_white',
    hovermode='x unified',
    height=500,
    width=800
)

fig.update_xaxes(range=[0, 5])
fig.show()